> 在线讲义：[第 3 章 · 分派器：按标签一层层处理](https://wikieden.github.io/pytorch-inside-out/lessons/03-dispatcher.html) · 本书《PyTorch 从里到外》

# 第 3 章 · 分派器：按标签一层层处理

本章回答：第 1 章说"分派器看标签"，标签到底有哪些、按什么顺序处理？混合精度、自动求导、换设备为什么能互不干扰地叠在一起？"meta 设备"是什么，为什么一个 4 TB 的张量不占内存？

配套讲义：仓库里 `pytorch-deep-dive/notes/lessons/03-dispatcher.md`（讲解、动画、练习都在讲义里）。对应原文：00-overview.md §2.2；ezyang「PyTorch internals」dispatch 一节、「Let's talk about the PyTorch dispatcher」（〔在线〕blog.ezyang.com/2020/09/）；本章所有输出都在本机 torch 2.15.0 上跑过（代码见本章末尾"代码对照"）

**怎么用**：在 `pytorch-deep-dive/solutions/chapter-code/` 下用 `uv run jupyter lab` 打开（或任何装了 torch 的环境）。从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。Mac 上能跑：计时类的格子在 Mac 上用 MPS，分布式的格子在本机起 2 个 Gloo 进程。

In [ ]:
import torch

torch.set_printoptions(precision=4, sci_mode=False)
print("准备好了", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 CPU 张量的标签 | 四个工种 | `keys(torch.randn(2))` | CPU、ADInplaceOrView、AutogradCPU、AutocastCPU |
| 第二步 autocast 里乘 | 先转 bf16 再记账 | `autocast_matmul()` | bf16、MmBackward0、上游 [None, ToCopyBackward0] |
| 第四步 Meta | 只算形状 | `meta_matmul()` | 名义 4 TB，形状 (10⁶, 10⁶) |
| 第四步 inference_mode | 没有 Autograd 标签 | `keys(z)` | DispatchKeySet(CPU, AutocastCPU) |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `autocast_matmul`**（来自 `ch03_dispatcher.py`）：autocast 里做 x @ w：返回 dtype、grad_fn 名字、上游节点名字。

In [ ]:
def autocast_matmul():
    """autocast 里做 x @ w：返回 dtype、grad_fn 名字、上游节点名字。"""
    torch.manual_seed(0)
    x = torch.randn(2, 3)
    w = torch.randn(3, 4, requires_grad=True)
    with torch.autocast("cpu", dtype=torch.bfloat16):
        y = x @ w
    ups = [None if f is None else type(f).__name__ for f, _ in y.grad_fn.next_functions]
    return y.dtype, type(y.grad_fn).__name__, ups

**函数 `keys`**（来自 `ch03_dispatcher.py`）：张量身上的 dispatch key（内部 API，只用来学习）。

In [ ]:
def keys(t: torch.Tensor) -> str:
    """张量身上的 dispatch key（内部 API，只用来学习）。"""
    return str(torch._C._dispatch_keys(t))

**函数 `meta_matmul`**（来自 `ch03_dispatcher.py`）：meta 设备上的 n × n 矩阵乘：只算形状，返回 (名义字节数, 结果形状)。

In [ ]:
def meta_matmul(n: int = 10**6):
    """meta 设备上的 n × n 矩阵乘：只算形状，返回 (名义字节数, 结果形状)。"""
    m = torch.empty(n, n, device="meta")
    return m.numel() * m.element_size(), tuple((m @ m.T).shape)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 看一眼真实的标签")
print("CPU 张量：", keys(torch.randn(2)))
if torch.backends.mps.is_available():
    print("MPS 张量：", keys(torch.randn(2, device="mps")))

In [ ]:
print("\n第二步 autocast → autograd → 厨房")
print("dtype / grad_fn / 上游：", autocast_matmul())

In [ ]:
print("\n第四步 Meta 与 inference_mode")
nbytes, shape = meta_matmul()
print(f"meta 张量名义上 {nbytes / 1e12:.0f} TB，m @ m.T 形状 {shape}")
with torch.inference_mode():
    z = torch.randn(2)
print("inference_mode 里的张量：", keys(z))

## 3. 代码里才看得到的

1. **普通张量一直带 Autograd 标签**：不要梯度时记账员只是放行（第 1 章的简化说法在这里补正）。
2. **`_dispatch_keys` 是下划线开头的内部 API**：只用来学习，不同版本输出顺序可能变。

## 4. 自己改一改

- 在 Mac 上看 `keys(torch.randn(2, device="mps"))`。
- 把 `autocast_matmul` 里的 dtype 换成 `torch.float16`，看 CPU 上会不会报错。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 张量身上贴标签，分派器按梯子往下传：
> 先混合精度，再记账，再版本登记，最后进厨房；
> 每层干完撕标签；加功能加一列，加算子加一行；Meta 只算形状不算数。